# Edmonton 311 Missed Waste Collection Analysis

This notebook prepares a reproducible analytical dataset for an interactive Tableau dashboard focused on City of Edmonton 311 requests related to missed residential waste collection.

**Business question:** Where and when are missed collection requests concentrated across Edmonton, and how do patterns vary by service type?

> Important: the metric is **311 requests**, not a verified count of unique missed pickups.


## 1. Imports and API helpers

The project uses two City of Edmonton Socrata endpoints: the 311 request dataset and the official neighbourhood boundary dataset.


In [2]:
from pathlib import Path
from io import StringIO

import pandas as pd
import requests
import geopandas as gpd
from shapely import wkt

URL_311 = "https://data.edmonton.ca/resource/q7ua-agfg.csv"
NEIGH_URL = "https://data.edmonton.ca/resource/65fr-66s6.csv"
PAGE_SIZE = 50_000


def fetch_socrata_csv(url, params, page_size=PAGE_SIZE):
    chunks = []
    offset = 0

    while True:
        page_params = dict(params)
        page_params["$limit"] = page_size
        page_params["$offset"] = offset

        response = requests.get(url, params=page_params, timeout=60)
        response.raise_for_status()
        chunk = pd.read_csv(StringIO(response.text))

        if chunk.empty:
            break

        chunks.append(chunk)
        if len(chunk) < page_size:
            break

        offset += page_size

    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()


## 2. Inspect the available 311 snapshot

This is a QA step. In the uploaded working notebook, the snapshot ranged from **2023-01-01** to **2026-10-02** with 3,458,556 total 311 records. Because the API is live, a rerun may return a different range.


In [3]:
date_range_311 = pd.read_csv(
    StringIO(
        requests.get(
            URL_311,
            params={
                "$select": "min(date_created) as earliest_request, max(date_created) as latest_request, count(*) as records"
            },
            timeout=60,
        ).text
    )
)
date_range_311


,earliest_request,latest_request,records
0,2023-01-01T00:04:03.000,2026-10-09T23:52:17.000,3473740


## 3. Define the missed-collection scope

Only six exact source descriptions are included. The analysis does not assume that other descriptions containing “missed” are operationally equivalent.


In [4]:
selected_missed = [
    "Missed Garbage Cart - Response Required",
    "Missed Garbage Cart - General Info",
    "Missed Res Collection General Info",
    "Missed Recycling - General Information",
    "Missed Yard Waste Response Required",
    "Missed Yard Waste General Information",
]

service_group_mapping = {
    "Missed Garbage Cart - Response Required": "Garbage",
    "Missed Garbage Cart - General Info": "Garbage",
    "Missed Res Collection General Info": "Residential General",
    "Missed Recycling - General Information": "Recycling",
    "Missed Yard Waste Response Required": "Yard Waste",
    "Missed Yard Waste General Information": "Yard Waste",
}


## 4. Download the selected 311 records

Rows without a neighbourhood are excluded from the spatial/neighbourhood analysis. The uploaded working notebook returned **42,530** such locatable selected requests.


In [5]:
description_sql = ",".join(
    "'" + value.replace("'", "''") + "'"
    for value in selected_missed
)

missed_raw = fetch_socrata_csv(
    URL_311,
    {
        "$select": "date_created, neighbourhood, service_description",
        "$where": (
            f"service_description in ({description_sql}) "
            "AND date_created IS NOT NULL "
            "AND neighbourhood IS NOT NULL"
        ),
    },
)

print(f"Selected rows: {len(missed_raw):,}")
missed_raw.head()


Selected rows: 42,838


,date_created,neighbourhood,service_description
0,2026-06-01T15:40:56.000,EDGEMONT,Missed Yard Waste Response Required
1,2026-04-10T10:31:17.000,HODGSON,Missed Garbage Cart - Response Required
2,2026-06-01T13:16:41.000,GREENFIELD,Missed Yard Waste General Information
3,2026-04-09T22:55:21.000,HODGSON,Missed Garbage Cart - Response Required
4,2026-04-10T08:11:21.000,INGLEWOOD,Missed Garbage Cart - Response Required


## 5. Clean dates and classify service groups


In [6]:
missed_raw["date_created"] = pd.to_datetime(
    missed_raw["date_created"], errors="coerce"
)

missed_raw = missed_raw.dropna(
    subset=["date_created", "neighbourhood", "service_description"]
).copy()

missed_raw["month_date"] = (
    missed_raw["date_created"].dt.to_period("M").dt.to_timestamp()
)
missed_raw["year"] = missed_raw["date_created"].dt.year
missed_raw["month"] = missed_raw["date_created"].dt.month
missed_raw["service_group"] = missed_raw["service_description"].map(
    service_group_mapping
)


## 6. Aggregate requests

The source-level table is first summarized by exact service description for QA, then by the dashboard grain of month × neighbourhood × service group.


In [8]:
qa_description_totals = (
    missed_raw.groupby("service_description")
    .size()
    .sort_values(ascending=False)
    .rename("requests")
)
qa_description_totals


service_description
Missed Garbage Cart - Response Required    37717
Missed Garbage Cart - General Info          3064
Missed Yard Waste Response Required          881
Missed Res Collection General Info           652
Missed Recycling - General Information       349
Missed Yard Waste General Information        175
Name: requests, dtype: int64

In [9]:
monthly_grouped = (
    missed_raw.groupby(
        ["month_date", "neighbourhood", "service_group"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "requests"})
)

monthly_grouped.head()


,month_date,neighbourhood,service_group,requests
0,2023-01-01,ALBERTA AVENUE,Garbage,3
1,2023-01-01,ALDERGROVE,Garbage,1
2,2023-01-01,ALLARD,Garbage,2
3,2023-01-01,AMBLESIDE,Garbage,3
4,2023-01-01,ARGYLL,Garbage,4


## 7. Build a complete monthly Tableau grid

A complete Month × Neighbourhood × Service Group grid is created so Tableau can display explicit zeros instead of gaps.

**Interpretation caution:** a zero means no matching selected request was observed in the extracted data for that combination. It does not prove that a particular source workflow/category was active in every period.


In [10]:
all_months = pd.date_range(
    monthly_grouped["month_date"].min(),
    monthly_grouped["month_date"].max(),
    freq="MS",
)

all_neighbourhoods = sorted(
    monthly_grouped["neighbourhood"].dropna().unique()
)

all_service_groups = [
    "Garbage",
    "Residential General",
    "Recycling",
    "Yard Waste",
]

full_monthly = pd.MultiIndex.from_product(
    [all_months, all_neighbourhoods, all_service_groups],
    names=["month_date", "neighbourhood", "service_group"],
).to_frame(index=False)

tableau_data = full_monthly.merge(
    monthly_grouped,
    on=["month_date", "neighbourhood", "service_group"],
    how="left",
)

tableau_data["requests"] = tableau_data["requests"].fillna(0).astype(int)
tableau_data["year"] = tableau_data["month_date"].dt.year
tableau_data["month"] = tableau_data["month_date"].dt.month
tableau_data["month_name"] = tableau_data["month_date"].dt.month_name()

print(f"Tableau grid rows: {len(tableau_data):,}")
print(f"Neighbourhood names: {tableau_data['neighbourhood'].nunique():,}")
print(f"Period: {tableau_data['month_date'].min().date()} to {tableau_data['month_date'].max().date()}")


Tableau grid rows: 58,880
Neighbourhood names: 320
Period: 2023-01-01 to 2026-10-01


## 8. Load official Edmonton neighbourhoods

The official neighbourhood dataset provides the current lookup fields and multipolygon geometry used by Tableau.


In [11]:
response = requests.get(
    NEIGH_URL,
    params={"$limit": 5000},
    timeout=60,
)
response.raise_for_status()
neigh_all = pd.read_csv(StringIO(response.text))

neigh_all["name_key"] = (
    neigh_all["name"].astype(str).str.strip().str.upper()
)
neigh_all["neighbourhood_number"] = (
    neigh_all["neighbourhood_number"].astype(str).str.strip()
)

print(neigh_all.shape)
neigh_all.head()


(407, 10)


,name,neighbourhood_number,descriptive_name,description,effective_start_date,effective_end_date,civic_ward_name,district,geometry_multipolygon,name_key
0,ABBOTTSFIELD,2010,Abbottsfield,"Abbottsfield was once owned by E.F. Carey, one...",1981-10-01T00:00:00.000,NaN,Métis,North Central,MULTIPOLYGON (((-113.38763361949056 53.5761348...,ABBOTTSFIELD
1,ALBANY,3460,Albany,NaN,2011-06-13T00:00:00.000,NaN,Anirniq,Northwest,MULTIPOLYGON (((-113.56024507936962 53.6316036...,ALBANY
2,ALBERTA AVENUE,1010,Alberta Avenue,One of the city's older residential neighbourh...,1981-10-01T00:00:00.000,NaN,Métis,North Central,MULTIPOLYGON (((-113.49225733739199 53.5766166...,ALBERTA AVENUE
3,ALBERTA PARK INDUSTRIAL,4010,Alberta Park Industrial,This non-residential neighbourhood is largely ...,1981-10-01T00:00:00.000,NaN,Nakota Isga,Jasper Place,MULTIPOLYGON (((-113.59041036545489 53.5665379...,ALBERTA PARK INDUSTRIAL
4,ALCES,6669,Alces,Formerly Decoteau North,2019-07-25T00:00:00.000,NaN,Sspomitapi,Ellerslie,MULTIPOLYGON (((-113.37648878753762 53.4248774...,ALCES


## 9. Reconcile historical/source neighbourhood names

The original 311 name is retained. A separate key is used only to connect records to the current official neighbourhood lookup.


In [12]:
name_aliases = {
    "CHAPPELLE AREA": "CHAPPELLE",
    "HORSE HILL NEIGHBOURHOOD 1A": "QUARRY RIDGE",
    "OLIVER": "WÎHKWÊNTÔWIN",
}

tableau_data["neighbourhood_key"] = (
    tableau_data["neighbourhood"].astype(str).str.strip().str.upper()
)
tableau_data["official_name_key"] = (
    tableau_data["neighbourhood_key"].replace(name_aliases)
)

lookup_current = (
    neigh_all[neigh_all["effective_end_date"].isna()][
        ["name_key", "neighbourhood_number"]
    ]
    .drop_duplicates()
    .rename(columns={"name_key": "official_name_key"})
)

tableau_data = tableau_data.merge(
    lookup_current,
    on="official_name_key",
    how="left",
)

tableau_data["geography_status"] = (
    tableau_data["neighbourhood_number"]
    .notna()
    .map({True: "Mapped", False: "No official polygon match"})
)

tableau_data["geography_status"].value_counts()


geography_status
Mapped                       58696
No official polygon match      184
Name: count, dtype: int64

In the uploaded notebook run, all current grid rows mapped except the 184 zero-filled grid rows associated with the single unmatched source name **LEWIS FARMS INDUSTRIAL**. The project deliberately leaves this unmatched rather than forcing it into a potentially incorrect current polygon.


## 10. Create current neighbourhood GeoJSON


In [13]:
current_neigh = neigh_all[
    neigh_all["effective_end_date"].isna()
    & neigh_all["geometry_multipolygon"].notna()
].copy()

current_neigh["geometry"] = current_neigh["geometry_multipolygon"].apply(wkt.loads)

neighbourhood_geo = gpd.GeoDataFrame(
    current_neigh[
        [
            "neighbourhood_number",
            "name",
            "descriptive_name",
            "civic_ward_name",
            "district",
            "geometry",
        ]
    ],
    geometry="geometry",
    crs="EPSG:4326",
)

print(neighbourhood_geo.shape)
print(neighbourhood_geo.geometry.geom_type.value_counts())


(407, 6)
MultiPolygon    407
Name: count, dtype: int64


## 11. Export Tableau-ready files

The outputs are written to the repository `data/` folders so the Tableau workbook can be connected using relative project structure.


In [14]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
spatial_dir = project_root / "data" / "spatial"
processed_dir.mkdir(parents=True, exist_ok=True)
spatial_dir.mkdir(parents=True, exist_ok=True)

export_columns = [
    "month_date",
    "year",
    "month",
    "month_name",
    "neighbourhood_number",
    "neighbourhood",
    "official_name_key",
    "service_group",
    "requests",
    "geography_status",
]

csv_path = processed_dir / "edmonton_missed_waste_collection_monthly.csv"
tableau_data[export_columns].to_csv(csv_path, index=False)

geojson_path = spatial_dir / "edmonton_neighbourhoods.geojson"
geojson_path.write_text(
    neighbourhood_geo.to_json(drop_id=True),
    encoding="utf-8",
)

print(f"CSV: {csv_path}")
print(f"GeoJSON: {geojson_path}")


CSV: C:\Users\nelub\data\processed\edmonton_missed_waste_collection_monthly.csv
GeoJSON: C:\Users\nelub\data\spatial\edmonton_neighbourhoods.geojson


## 12. Final QA

These checks confirm the analytical grain, service totals, date range, and geography coverage before opening Tableau.


In [15]:
print("Rows:", len(tableau_data))
print("Neighbourhoods:", tableau_data["neighbourhood"].nunique())
print("Period:", tableau_data["month_date"].min(), "to", tableau_data["month_date"].max())
print()
print("Request totals by service group:")
print(tableau_data.groupby("service_group")["requests"].sum().sort_values(ascending=False))
print()
print("Geography status:")
print(tableau_data["geography_status"].value_counts())


Rows: 58880
Neighbourhoods: 320
Period: 2023-01-01 00:00:00 to 2026-10-01 00:00:00

Request totals by service group:
service_group
Garbage                40781
Yard Waste              1056
Residential General      652
Recycling                349
Name: requests, dtype: int64

Geography status:
geography_status
Mapped                       58696
No official polygon match      184
Name: count, dtype: int64


## 13. Tableau notes

The Tableau dashboard is built from the exported CSV and GeoJSON using a Many-to-One relationship on `neighbourhood_number`.

See [`../TABLEAU_SETUP.md`](../TABLEAU_SETUP.md) for the map, Top 10, monthly trend, filters, calculated field, and dashboard action configuration.
